# Validate best.pt from interrupted run
**No GPU needed** — runs on CPU runtime

1. Runtime → Change runtime type → None (CPU)
2. Run all
3. Results saved inside existing Drive checkpoints folder — no clutter

In [ ]:
# Mount Drive and find best.pt
from google.colab import drive
drive.mount('/content/drive')
import os

checkpoint_dir = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'best.pt' in files:
        checkpoint_dir = root
        break

if checkpoint_dir is None:
    raise FileNotFoundError('best.pt not found in Drive')

best_pt = os.path.join(checkpoint_dir, 'best.pt')
print(f'Found: {best_pt}')
print(f'Size: {os.path.getsize(best_pt)/1e6:.1f} MB')
print('Checkpoint folder contents:', os.listdir(checkpoint_dir))


In [ ]:
# Install ultralytics
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'ultralytics>=8.4.0'], check=True)
import ultralytics
print(f'ultralytics {ultralytics.__version__}')


In [ ]:
# Extract dataset
import tarfile, os, yaml

tar_path = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f == 'backup_merged_dataset.tar.gz':
            tar_path = os.path.join(root, f)
            break
    if tar_path:
        break

if tar_path is None:
    raise FileNotFoundError('backup_merged_dataset.tar.gz not found in Drive')

print(f'Extracting {tar_path}...')
extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)
with tarfile.open(tar_path) as tf:
    tf.extractall(extract_dir)

data_yaml = os.path.join(extract_dir, 'merged_dataset', 'data.yaml')
base = os.path.join(extract_dir, 'merged_dataset')
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = base
cfg['train'] = os.path.join(base, 'train', 'images')
cfg['val'] = os.path.join(base, 'val', 'images')
cfg['test'] = os.path.join(base, 'test', 'images')
with open(data_yaml, 'w') as f:
    yaml.dump(cfg, f)
print('Dataset ready')


In [ ]:
# Run validation on best.pt and save results to Drive checkpoints folder
from ultralytics import YOLO
import json, os

print(f'Loading {best_pt}...')
model = YOLO(best_pt)

print('Running validation on test split (CPU, takes ~10-20 min)...')
val_results = model.val(
    data=data_yaml,
    split='test',
    imgsz=640,
    batch=8,
    device='cpu',
    plots=False,
)

box = val_results.box
names = val_results.names

per_class = {}
for idx, name in names.items():
    if idx < len(box.ap50):
        per_class[name] = float(box.ap50[idx])

f1 = 2 * float(box.mp) * float(box.mr) / (float(box.mp) + float(box.mr) + 1e-9)

results = {
    'run_id': 'run_fast_yolo26s_epoch65',
    'model_variant': 'yolo26s',
    'checkpoint': 'best.pt (epoch ~65, interrupted)',
    'completed': False,
    'map50': float(box.map50),
    'map50_95': float(box.map),
    'precision': float(box.mp),
    'recall': float(box.mr),
    'f1': f1,
    'per_class_map50': per_class,
    'dataset': 'merged Bird/Drone/UAV 31k images, 30% fraction for training',
    'epochs_completed': 65,
    'hardware': 'Colab T4',
}

print('\n=== VALIDATION RESULTS ===')
print(f'mAP@0.5:      {results["map50"]:.4f}')
print(f'mAP@0.5:0.95: {results["map50_95"]:.4f}')
print(f'Precision:    {results["precision"]:.4f}')
print(f'Recall:       {results["recall"]:.4f}')
print(f'F1:           {results["f1"]:.4f}')
print('Per-class mAP@0.5:')
for cls, v in per_class.items():
    print(f'  {cls}: {v:.4f}')

# Save inside existing checkpoints folder — no new folders created
out_path = os.path.join(checkpoint_dir, 'validation_results_run1_yolo26s.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: {out_path}')
